# Step 0 — 内存、磁盘与实验配置

对应 `reference/plan.md` 的 Step 0。本 Notebook 不删除任何缓存或模型，先回答：

- 系统内存、GPU 显存、磁盘是否足以开始；
- 是否有遗留进程占用 GPU；
- 建立后续所有 Notebook 共用的 3090/L4 单卡配置。

**通过标准：** CUDA 可用、GPU 显存约 24GB、磁盘至少预留 15GB、系统可用内存至少 12GB。

In [ ]:
# @title Step 00.1 — 初始化运行环境
from pathlib import Path
import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")
REPO_URL = "https://github.com/wtree101/ZIP-RC-Colab.git"
REPO_BRANCH = "main"
SYNC_REPO = True

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)
elif SYNC_REPO:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

if not ZIP_PY.exists():
    raise FileNotFoundError(
        f"未找到 {ZIP_PY}。请先建立 ZIP-RC 的 mamba 环境，再重新运行本 Notebook。"
    )

kernel_required = ["torch", "numpy", "pandas", "pyarrow", "matplotlib", "sklearn", "psutil"]
kernel_missing = [name for name in kernel_required if importlib.util.find_spec(name) is None]
if kernel_missing:
    raise ModuleNotFoundError(f"Colab kernel 缺少可视化依赖: {kernel_missing}")

env_check = subprocess.run(
    [
        str(ZIP_PY),
        "-c",
        (
            "import importlib.util, json; "
            "mods=['torch','vllm','transformers','datasets','pandas','pyarrow']; "
            "print(json.dumps([m for m in mods if importlib.util.find_spec(m) is None]))"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
env_missing = json.loads(env_check.stdout.strip())
if env_missing:
    raise ModuleNotFoundError(f"zip mamba 环境缺少依赖: {env_missing}")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import torch
from IPython.display import display

sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import (
    gate,
    gate_frame,
    load_config,
    model_artifacts_exist,
    read_jsonl,
    require_columns,
    rolling_edges,
    run_repo,
    save_stage_report,
)

print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)

In [ ]:
# @title Step 00.2 — 实验进度与下一步
stage_names = {
    "00": "环境与配置",
    "01": "Pilot 生成",
    "02": "Pilot 质量",
    "03": "正式训练数据",
    "04": "Prompt split",
    "05": "Predictor 训练",
    "06": "Predictor 评估",
    "07": "Controller 对比",
    "08": "毕业决策",
}
report_directory = REPO / "artifacts" / "stage_reports"
progress_rows = []
for stage, name in stage_names.items():
    matches = sorted(report_directory.glob(f"{stage}_*.json")) if report_directory.exists() else []
    report = json.loads(matches[-1].read_text(encoding="utf-8")) if matches else None
    operational_ok = bool(report and report.get("operational_passed"))
    scientific_ok = bool(report and report.get("scientific_passed"))
    complete = operational_ok and scientific_ok
    status = "✅ 完成" if complete else "⚠️ 需检查" if report else "⬜ 未运行"
    progress_rows.append(
        {
            "Step": stage,
            "阶段": name,
            "状态": status,
            "operational": operational_ok if report else None,
            "scientific": scientific_ok if report else None,
            "report": matches[-1].name if matches else "—",
            "complete": complete,
        }
    )

progress = pd.DataFrame(progress_rows)
display(progress.drop(columns="complete"))
completed_count = int(progress["complete"].sum())
next_rows = progress[~progress["complete"]]
next_step = str(next_rows.iloc[0]["Step"]) if not next_rows.empty else None

fig, ax = plt.subplots(figsize=(9, 1.25))
ax.barh(["ZIP-RC"], [completed_count], color="#49beaa")
ax.barh(["ZIP-RC"], [len(progress) - completed_count], left=[completed_count], color="#e5e7eb")
ax.set(xlim=(0, len(progress)), xlabel="completed stages")
ax.text(completed_count / 2 if completed_count else 0.15, 0, f"{completed_count}/{len(progress)}", va="center")
plt.tight_layout()
plt.show()

if next_step is None:
    print("✅ Step 0–8 均已完成；查看 Step 08 的毕业结论。")
else:
    print(f"下一步：从左侧 Outline 跳到 Step {int(next_step)}。完成后重新运行本面板。")

In [ ]:
# @title Step 00.3 — 检查 RAM、磁盘和 GPU
# 安全清理当前 Notebook 自己不再引用的 Python/CUDA 缓存；不会删除文件，也不会终止其他进程。
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

vm = psutil.virtual_memory()
disk = shutil.disk_usage(REPO)
cuda_ok = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "No CUDA GPU"
gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 2**30 if cuda_ok else 0.0
gpu_free_gb = torch.cuda.mem_get_info(0)[0] / 2**30 if cuda_ok else 0.0

resources = pd.DataFrame(
    [
        {"resource": "System RAM", "used_gb": (vm.total - vm.available) / 2**30, "free_gb": vm.available / 2**30},
        {"resource": "Disk", "used_gb": (disk.total - disk.free) / 2**30, "free_gb": disk.free / 2**30},
        {"resource": "GPU VRAM", "used_gb": gpu_total_gb - gpu_free_gb, "free_gb": gpu_free_gb},
    ]
)
display(resources.round(2))

ax = resources.set_index("resource")[["used_gb", "free_gb"]].plot(
    kind="barh", stacked=True, figsize=(9, 3.5), color=["#ef767a", "#49beaa"]
)
ax.set_xlabel("GB")
ax.set_title("开始实验前的资源占用")
ax.legend(["已用", "可用"], loc="lower right")
plt.tight_layout()
plt.show()

def directory_size_gb(path):
    if not path.exists():
        return 0.0
    return sum(item.stat().st_size for item in path.rglob("*") if item.is_file()) / 2**30

storage_roots = {
    "repo/data": REPO / "data",
    "repo/models": REPO / "models",
    "HF cache": Path.home() / ".cache" / "huggingface",
    "torch cache": Path.home() / ".cache" / "torch",
    "pip cache": Path.home() / ".cache" / "pip",
}
storage = pd.Series({name: directory_size_gb(path) for name, path in storage_roots.items()}).sort_values()
display(storage.rename("GB").to_frame().round(2))
storage.plot.barh(figsize=(9, 3.5), color="#f2cf5b", title="常见数据/模型缓存占用（只读检查）")
plt.xlabel("GB")
plt.tight_layout()
plt.show()

print({"gpu": gpu_name, "vram_gb": round(gpu_total_gb, 1), "bf16": torch.cuda.is_bf16_supported() if cuda_ok else False})
try:
    apps = subprocess.run(
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader"],
        check=False, capture_output=True, text=True,
    ).stdout.strip()
    print("GPU compute processes:\n", apps or "无正在运行的 compute process")
except FileNotFoundError:
    print("nvidia-smi 不可用；跳过进程列表。")

In [ ]:
# @title Step 00.4 — 编辑并保存实验配置
# 单张 RTX 3090 / Colab L4 的默认正式小实验。后续只需修改这里并重新运行本 cell。
config = {
    "experiment_name": "qwen3_0.6b_ziprc_small_validation",
    "model_id": "Qwen/Qwen3-0.6B",
    "grader_model_id": "Qwen/Qwen2.5-Math-1.5B-Instruct",
    "dataset": "rohinm/adaptivemath",
    "split": "train",
    "prompt_column": "problem",
    "answer_column": "answer",
    "dtype": "bfloat16",
    "distribution_token_id": 151669,
    "num_length_bins": 8,
    "reward_values": [0.0, 1/6, 2/6, 3/6, 4/6, 5/6, 1.0],
    "generation_max_model_len": 4096,
    "max_output_tokens": 2048,
    "max_num_seqs": 2,
    "temperature": 1.0,
    "min_p": 0.1,
    "pilot_prompts": 200,
    "pilot_rollouts_per_prompt": 2,
    "training_prompts": 2000,
    "training_rollouts_per_prompt": 2,
    "train_fraction": 0.8,
    "validation_fraction": 0.1,
    "test_fraction": 0.1,
    "stage1_steps": 500,
    "stage2_steps": 800,
    "num_epochs": 3,
    "batch_size": 1,
    "gradient_accumulation_steps": 8,
    "stage1_learning_rate": 1e-4,
    "stage2_learning_rate": 5e-5,
    "train_max_length": 4096,
    "grader_max_model_len": 4096,
    "gpu_memory_utilization": 0.85,
    "controller_rollouts_per_prompt": 4,
    "paths": {
        "pilot": "data/pilot_rollouts.parquet",
        "full": "data/experiment_rollouts.parquet",
        "train": "data/splits/train.parquet",
        "validation": "data/splits/validation.parquet",
        "test": "data/splits/test.parquet",
        "train_value": "data/splits/train_with_value.parquet",
        "intermediate_model": "models/experiment_joint_correct",
        "final_model": "models/experiment_ziprc_final",
        "stage1_metrics": "artifacts/metrics/stage1.jsonl",
        "stage2_metrics": "artifacts/metrics/stage2.jsonl",
        "predictor_positions": "artifacts/predictor_positions.parquet",
        "predictor_metrics": "artifacts/predictor_metrics.json",
        "controller_rollouts": "data/controller_rollouts.parquet",
        "controller_scored": "data/controller_scored.parquet",
    },
}

for directory in [REPO / "data" / "splits", REPO / "models", REPO / "artifacts" / "metrics", REPO / "artifacts" / "stage_reports"]:
    directory.mkdir(parents=True, exist_ok=True)
config_path = REPO / "artifacts" / "experiment_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
display(pd.DataFrame([config]).T.rename(columns={0: "value"}))
print("Saved:", config_path)

In [ ]:
# @title Step 00.5 — Step 0 验收
checks = [
    gate("CUDA 可用", cuda_ok, gpu_name),
    gate("24GB 级 GPU", gpu_total_gb >= 20, f"检测到 {gpu_total_gb:.1f} GB；目标为 RTX 3090/L4"),
    gate("系统可用内存", vm.available / 2**30 >= 12, f"可用 {vm.available / 2**30:.1f} GB"),
    gate("磁盘空间", disk.free / 2**30 >= 15, f"可用 {disk.free / 2**30:.1f} GB"),
    gate("BF16 可用", bool(cuda_ok and torch.cuda.is_bf16_supported()), "配置固定使用 BF16"),
    gate("配置已保存", config_path.exists(), str(config_path)),
]
display(gate_frame(checks))
report = save_stage_report(
    REPO,
    "00_memory_and_config",
    checks,
    {"gpu": gpu_name, "gpu_total_gb": gpu_total_gb, "ram_available_gb": vm.available / 2**30, "disk_free_gb": disk.free / 2**30},
)
print("Stage report:", report)